# Prompt Engineering with Python using LLMs

**Master BDCC — Université Mohammed VI Polytechnique**

---

## Introduction

This notebook demonstrates a complete prompt engineering workflow:
1. Installation & Setup
2. Dataset loading and Exploratory Data Analysis (EDA)
3. Multiple prompting techniques applied to NLP tasks
4. Evaluation with Accuracy, Precision, Recall, F1 Score
5. Comparison charts across techniques

**Supported LLM Providers:** OpenAI, Ollama, Groq

## 1. Installation

Run these commands in your terminal before executing this notebook:

```bash
cd PromptEngineeringProject
python -m venv venv
venv\Scripts\activate        # Windows
pip install -r requirements.txt
copy .env.example .env        # Then add your API keys
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import get_settings
from utils.logger import setup_logging

setup_logging()
settings = get_settings()
print(f'Project root: {settings.project_root}')

## 2. Initialize LLM Client

Choose your provider: `openai`, `ollama`, or `groq`

In [ ]:
PROVIDER = 'openai'  # Change to 'ollama' or 'groq'

if PROVIDER == 'openai':
    from llms.openai_client import OpenAIClient
    client = OpenAIClient()
elif PROVIDER == 'ollama':
    from llms.ollama_client import OllamaClient
    client = OllamaClient()
elif PROVIDER == 'groq':
    from llms.groq_client import GroqClient
    client = GroqClient()

print(f'Provider: {client.provider_name} | Model: {client.model}')

## 3. Dataset Loading — IMDB Sentiment Analysis

In [ ]:
from datasets.imdb_loader import IMDBLoader

loader = IMDBLoader()
df = loader.load(sample_size=20)
df = loader.clean(df)

print('Dataset Statistics:')
stats = loader.statistics(df)
for k, v in stats.items():
    print(f'  {k}: {v}')

df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Sentiment Distribution')

df['text_length'] = df['text'].str.len()
for label, color in [('positive', '#2ecc71'), ('negative', '#e74c3c')]:
    subset = df[df['label'] == label]['text_length']
    axes[1].hist(subset, bins=10, alpha=0.6, label=label, color=color)
axes[1].set_title('Text Length Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Prompt Engineering — Single Example Demo

In [ ]:
from prompts.zero_shot_prompt import ZeroShotPrompt, SENTIMENT_ZERO_SHOT
from prompts.few_shot_prompt import FewShotPrompt, SENTIMENT_TASK, SENTIMENT_FEW_SHOT_EXAMPLES
from prompts.chain_of_thought_prompt import ChainOfThoughtPrompt, SENTIMENT_COT_TASK

test_text = df.iloc[0]['text']
print(f'Test review: {test_text[:100]}...')
print(f'True label: {df.iloc[0]["label"]}\n')

# Zero-Shot
zs = ZeroShotPrompt.run(client, SENTIMENT_ZERO_SHOT, test_text)
print(f'Zero-Shot: {zs.content.strip()} ({zs.execution_time:.2f}s)')

# Few-Shot
fs = FewShotPrompt.run(client, SENTIMENT_TASK, SENTIMENT_FEW_SHOT_EXAMPLES, test_text)
print(f'Few-Shot: {fs.content.strip()} ({fs.execution_time:.2f}s)')

# Chain of Thought
cot = ChainOfThoughtPrompt.run(client, SENTIMENT_COT_TASK, test_text)
print(f'CoT: {ChainOfThoughtPrompt.extract_answer(cot.content)} ({cot.execution_time:.2f}s)')

## 6. Full Evaluation — All Techniques

In [ ]:
from use_cases.sentiment_analysis import SentimentAnalysisUseCase

use_case = SentimentAnalysisUseCase(client)
comparison = use_case.run_all_techniques(df)
comparison

## 7. Comparison Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

comparison.plot(x='Technique', y='Accuracy', kind='bar', ax=axes[0], color='#3498db', legend=False)
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis='x', rotation=45)

comparison.plot(x='Technique', y='Avg Time (s)', kind='bar', ax=axes[1], color='#e67e22', legend=False)
axes[1].set_title('Average Execution Time (s)')
axes[1].tick_params(axis='x', rotation=45)

comparison.plot(x='Technique', y='Total Cost ($)', kind='bar', ax=axes[2], color='#2ecc71', legend=False)
axes[2].set_title('Total Cost Estimate ($)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(settings.output_dir / 'notebook_comparison.png', dpi=150)
plt.show()

## 8. Spam Detection Use Case

In [ ]:
from use_cases.spam_detection import SpamDetectionUseCase
from datasets.spam_loader import SpamLoader

spam_loader = SpamLoader()
spam_df = spam_loader.load(sample_size=20)
spam_df = spam_loader.clean(spam_df)

spam_use_case = SpamDetectionUseCase(client)
spam_comparison = spam_use_case.run_all_techniques(spam_df)
spam_comparison

## 9. Gold Examples

In [ ]:
from use_cases.gold_examples import get_gold_examples

gold = get_gold_examples('sentiment')
gold_df = pd.DataFrame(gold)
print('Gold Examples are manually labeled reference data used for:')
print('  - Benchmarking (ground truth)')
print('  - Few-shot learning (demonstration examples)')
print('  - Reproducible evaluation\n')
gold_df

## 10. Conclusion

This notebook demonstrated:
- Loading and analyzing NLP datasets (IMDB, SMS Spam)
- Applying 6+ prompt engineering techniques
- Evaluating with standard ML metrics
- Comparing accuracy, speed, and cost

**Key Findings:**
- Few-Shot and Role Prompting typically achieve highest accuracy
- Zero-Shot provides a strong baseline at lowest cost
- Chain of Thought improves interpretability
- Self-Consistency reduces variance at higher cost

For the full CLI experience, run: `python main.py`